_**Initialization**_


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window 

_**Read the Bronze table into silver layer**_

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

# _**Trimming**_

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,trim(col(field.name)))
df.display()

In [0]:
df = df.withColumn(
    "cat_id",
    F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_")
)


_**Product Key Parsing**_

In [0]:


df = df.withColumn(
    "prd_key",
    F.substring(col("prd_key"), 7, F.length(col("prd_key")))
)


# **_Replace Null Value to 0 using coalesce_**

In [0]:
df = df.withColumn("prd_cost",F.coalesce(col('prd_cost'),F.lit(0)))

# **_Product Line Normalization_**

In [0]:
df = (
    df.withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M",'Mountain')
         .when(F.upper(col("prd_line")) == "R",'Road')
         .when(F.upper(col("prd_line")) == "S","Other Sales")
         .when(F.upper(col("prd_line")) == "T","Touring")
         .otherwise("n/a")
    )
)

# **_Date Casting_**

In [0]:
df = df.withColumn("prd_start_dt",col("prd_start_dt").cast(DateType()))

# **_Renaming Column_**

In [0]:
rename_map ={
    "prd_id":"Product_id",
    "cat_id":"Category_id",
    "prd_key":"Product_number",
    "prd_nm":"Product_name",
    "prd_cost":"Product_cost",
    "prd_line":"Product_line",
    "prd_start_dt":"start_date",
    "prd_end_dt":"end_date"
}
for old_name,new_name in rename_map.items():
    if old_name in df.columns:
        df = df.withColumnRenamed(old_name,new_name)

In [0]:

df.limit(10).display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver.crm_product")